# Field-only: can the second stage alone place B in A's frame, starting from a random guess?

## Description

<!-- Written before the first tracked run (2026-10-26, ~13:30). Criteria drafted by Claude (autonomous); issue #14.
Follows notebook 06 (order-only GW, NOT USABLE). Disclosure: a scratch CPU pilot (64 pairs, gd with 4 starts; the
learned methods trained for 400 steps only) ran before this description was written, to check that the setup is
not degenerate; the thresholds below are the same as notebook 06's (0.05 in A's frame) and were not tuned on it. -->

### The problem, standalone

Two related tasks share the search space [0, 1]. **A** is scarce: n_A observations (x_a, y_a). **B** is dense:
N_B observations (u_i, y_i) of a *transformed* copy of A's function, f_B(u) = f_A(T(u)), with T an unknown increasing
map (here the output map h is the identity; each task is still normalized on its own window, so B's values are an
affine-rescaled copy of A's). If T were known, every B point could be placed in A's frame at
**B_inA_i = (T(u_i), y_i on A's scale)** and used as extra data for A. *Registration* is recovering this placement
from the data alone.

The planned registration model has two stages: a front half that finds a few landmark matches (tested alone in
notebook 06: order-only GW was not precise enough) and a **second stage that moves a dense field**, one position and
value per B point, until the placed B agrees with A's few observations. This notebook tests the second stage
**in isolation and without any landmarks**: the field starts from a random guess. If it can register from there, GW
is not needed as a seed; if it can only fix small errors, the seed has to come from somewhere else.

### Prior (data)

Phase-1 pair prior (`experimental/regpfn/prior.py`), d = 1, the same pairs as notebook 06: f_A is a random BNN draw
(pfn-repro's BNN prior, evaluated on all of R so that shifted windows stay defined); T(u) =
Kumaraswamy-warp(0.5 + a (u + s − 0.5)) with shift s ~ U[−0.3, 0.3], scale a ~ logU[0.7, 1.4], warp log α, log β ~
N(0, 0.35²). Cells: **C00** (T = id: a sanity stratum) and **C10** (T on: the question). Each task is normalized
affinely onto [0.1, 0.9] over its own window; observation noise σ ~ logU[1e-3, 3e-2]. N_B = 64, n_A ∈ {2, 5, 10, 20}.
Evaluation: 256 fixed pairs per cell, nested in n_A; 128 query points of A per pair (noise-free f_A known).

### The field and what "registered" means

A field assigns every B point i a position and value in A's frame, (x̂_i, ŷ_i). The truth is B_inA. **Field
error** err_x = RMS over B points inside A's window of |x̂_i − T(u_i)| (in A's frame; err_y likewise). A pair counts
as **registered** if err_x < 0.05.

**Starts.** *random*: an affine guess drawn from the prior's own shift and scale ranges, independently of the truth,
x̂⁰(u) = 0.5 + a₀ (u + s₀ − 0.5), s₀ ~ U[−0.3, 0.3], a₀ ~ logU[0.7, 1.4], ŷ⁰ = y_B (no warp guess). *identity*:
x̂⁰ = u (in C00 this *is* the truth: a no-harm check). Every method gets the same start per pair.

**Data fit.** The placed B, rendered at x as a soft nearest neighbour,
r_ℓ(x) = Σ_i softmax_i(−|x − x̂_i|² / 2ℓ²) ŷ_i, and the energy E_ℓ = (1/n_A) Σ_a (y_a − r_ℓ(x_a))²: how well
the placed B explains A's points. ℓ sets how far a B point "sees".

### Methods

All three move the same field; the learned ones are weight-tied operators (the same layer applied T = 12 times).

- **`gd`, no learning (reference).** The classical way: fit a monotone piecewise-linear map of u (8 segments,
  positive slopes) and an affine value map ŷ = α y_B + β by gradient descent (Adam, 300 steps) on E_ℓ, with ℓ
  annealed from 0.15 to 0.02 (coarse to fine: a wide kernel first gives the far-off field a direction, a narrow one
  then makes it precise). From the shared start (`gd`); the best energy of 8 starts (`gd_best8`, the shared one + 7
  random); and the start that ends closest to the truth (`gd_oracle8`, not available in practice: a search ceiling).
- **`operator`: the phase-1 refinement layer as it is** (`RegBack.refine` with explicit residuals, steps ≤ 0.1, the
  variant of notebook 04 §2). Per iteration each B point (1) attends to A's points near its current x̂ with a
  distance-biased attention, and gets the kernel-weighted local residuals (y_a − ŷ_i, x_a − x̂_i); (2) a GRU updates
  its state; (3) proposes a step, scaled by tanh of the local evidence mass Σ_a exp(−|x̂_i − x_a|² / 2ℓ_e²); (4) the
  steps are smoothed over B neighbours in B's own frame (so the field moves as a sheet). Its evidence is local
  by design: a B point far from all of A's points does not move.
- **`optimizer`: a learned optimizer on the same field (new).** The same state / GRU / B-frame smoothing, but each
  iteration also feeds the GRU the **gradient of the data fit**, ∂E_ℓ/∂x̂_i and ∂E_ℓ/∂ŷ_i, at three widths
  ℓ ∈ {0.02, 0.05, 0.15} (each divided by its RMS over B, plus the log RMS), and the B-mean of these gradients; it
  proposes a local step (smoothed in B's frame) plus a global step from the pooled state (so a global shift needs no
  slow propagation); steps ≤ 0.1, no evidence gate. It is told *which way is downhill*, at several scales, and
  learns how far to go and how to combine scales, i.e. what `gd` does by hand.
- **References:** `start` (the field before moving) and `truth` (B_inA itself: err 0; its pred_rmse is the
  best a perfectly registered B can do for A's queries at N_B = 64).

**Training of the learned methods** (isolated component test, privileged supervision): the loss is the field error
itself, Σ_t 0.8^(T−t) |(x̂_t, ŷ_t) − B_inA|² over inside-window B points, at every iteration t (as in notebook 04).
6,000 steps, batch 32, Adam (lr 1e-3, warm-up, cosine); per step a cell (C10 with p = 0.8, else C00), n_A ~ U{2..20},
the random start (identity with p = 0.1). No read-out, no NLL: only "can the operator move the field to the truth".
Training curves (field MSE, validation share / median at n_A = 5, 10 on 128 fixed C10 pairs) go to MLflow.

### What to look for

1. Share registered and median err_x per method, cell, n_A and start (table and curves, 95% bootstrap CIs over pairs).
2. Learned vs `gd`: does the learned operator beat the hand-made optimizer on the same energy?
3. **Failure type**: among unregistered pairs, the share with E(fit) ≤ E(truth). Then the fit explains A's points as
   well as the truth does, so with this n_A the data cannot pick the truth (an identifiability limit, not the
   operator's fault). Otherwise it is a search failure (the truth was better and was not found).
4. pred_rmse (the rendered B as a predictor of f_A at A's queries): does a field with err_x ≥ 0.05 still predict well
   (e.g. errors in flat regions)?
5. 1d examples (C10, n_A = 5 and 10): the field's trajectory over iterations from the random start.

### Rule (pre-registered; `analysis/fieldonly.py::verdict`)

At **C10, n_A = 10, random start**, for the best learned method:
- **USABLE** if share registered ≥ 0.50 and median err_x ≤ 0.05;
- **PARTIAL** if not USABLE, but its share's CI lies above the `start`'s and its median's CI below the `start`'s
  (it reliably moves the field toward the truth, without reaching it);
- **NOT USABLE** otherwise.
Also reported: **learned beats gd** if share ≥ `gd_best8` + 0.10 at n_A = 10 and 20; the **C00 no-harm check**
(identity start = truth: median err_x stays ≤ 0.02 at n_A = 10). A USABLE second stage means the registration model
can drop GW as a seed and start from a random (or few random) field(s); NOT USABLE with mostly data-consistent
failures means the bottleneck is information (n_A), not the operator.

### Limits

d = 1, h = id (isolation, decision 0003: output warps are not tested here), N_B = 64, one seed per learned method,
one training budget (6,000 steps; notebook 06's solvers saturated after 1,000–2,000). The field is supervised with
privileged B_inA: it measures what the operator *can* do, not what the full model would learn from the NLL alone.
The `gd` reference uses a monotone, piecewise-linear model class (T is in it up to the warp's curvature); the
learned methods move per-point fields (more flexible, no monotonicity).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meta_pfn.analysis import load_snapshot, show_header
from meta_pfn.analysis.fieldonly import summary, verdict

SNAPSHOTS = {  # one sweep per method (launched separately on ulysses)
    "gd": "results/fieldonly/<sweep_id>",
    "operator": "results/fieldonly/<sweep_id>",
    "optimizer": "results/fieldonly/<sweep_id>",
}
snaps = {k: load_snapshot(v) for k, v in SNAPSHOTS.items()}
for k, s in snaps.items():
    print(f"== {k}: {s.manifest['status_counts']} (DONE / FAILED / STARTED / missing), host {s.manifest['host']}")
show_header(snaps["optimizer"])

## Results

### Field quality per method, cell, n_A and start (pairs; 95% bootstrap CIs over pairs)

In [ ]:
ev = pd.concat([s.table("eval") for s in snaps.values()], ignore_index=True)
sm = summary(ev)
order = ["start", "gd", "gd_best8", "gd_oracle8", "operator", "optimizer", "truth"]
sm["method"] = pd.Categorical(sm.method, order)
pd.set_option("display.width", 220)
cols = ["cell", "init", "n_a", "method", "share", "share_lo", "share_hi", "med_err_x", "med_lo", "med_hi", "med_err_y",
        "med_pred", "fail_consistent"]
sm.sort_values(["cell", "init", "n_a", "method"])[cols].round(3)

In [ ]:
# fixed categorical order (gd = slot 1, operator = 2, optimizer = 3); line style / marker as secondary encoding
STYLE = {
    "start": dict(color="#8a8a85", ls=":", marker="x", label="start (no move)"),
    "gd": dict(color="#2a78d6", ls="-", marker="o", label="gd (shared start)"),
    "gd_best8": dict(color="#2a78d6", ls="--", marker="s", label="gd best energy of 8 starts"),
    "gd_oracle8": dict(color="#2a78d6", ls=":", marker="^", label="gd oracle of 8 (ceiling)"),
    "operator": dict(color="#eb6834", ls="-", marker="D", label="operator (phase-1 layer)"),
    "optimizer": dict(color="#1baf7a", ls="-", marker="P", label="optimizer (learned, gradient-fed)"),
}


def panel_grid(metric, lo, hi, ylabel, log=False, ref=None):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
    for i, init in enumerate(["random", "identity"]):
        for j, cell in enumerate(["C00", "C10"]):
            ax = axes[i, j]
            for m, st in STYLE.items():
                d = sm.loc[(sm.cell == cell) & (sm.init == init) & (sm.method == m)].sort_values("n_a")
                if d.empty:
                    continue
                ax.plot(d.n_a, d[metric], lw=2, ms=6, **st)
                ax.fill_between(d.n_a, d[lo], d[hi], color=st["color"], alpha=0.12, lw=0)
            if ref is not None:
                ax.axhline(ref, color="0.3", lw=0.6, ls="-")
            ax.set_title(f"{cell} ({'T = id' if cell == 'C00' else 'T on'}), {init} start", fontsize=10)
            ax.set_xscale("log")
            ax.set_xticks([2, 5, 10, 20], ["2", "5", "10", "20"])
            ax.grid(alpha=0.25, lw=0.5)
            if log:
                ax.set_yscale("log")
            if j == 0:
                ax.set_ylabel(ylabel)
            if i == 1:
                ax.set_xlabel("n_A")
    axes[0, 0].legend(fontsize=8, frameon=False)
    fig.tight_layout()


panel_grid("share", "share_lo", "share_hi", "share of pairs registered (err_x < 0.05)")

In [ ]:
panel_grid("med_err_x", "med_lo", "med_hi", "median field error err_x (A's frame)", log=True, ref=0.05)

### Failure type: identifiability or search?

Among unregistered pairs (err_x ≥ 0.05), the share whose fit explains A's n_A points at least as well as the truth
(E_0.02(fit) ≤ E_0.02(truth)). High = the data cannot pick the truth at this n_A; low = the truth was better and was
not found.

In [ ]:
fc = sm.loc[(sm.init == "random") & sm.method.isin(["gd", "gd_best8", "operator", "optimizer"])]
fc.pivot_table(index=["cell", "n_a"], columns="method", values="fail_consistent", observed=True).round(2)

### Does the placed B predict A? (pred_rmse at A's queries inside B's image; median over pairs)

`truth` is the ceiling at N_B = 64 (B_inA rendered with the same soft nearest neighbour); `start` is the unmoved
guess.

In [ ]:
pp = sm.loc[sm.init == "random"].pivot_table(index=["cell", "n_a"], columns="method", values="med_pred", observed=True)
pp.round(3)

### Training curves of the learned methods (validation: 128 fixed C10 pairs, random start; also in MLflow)

In [ ]:
gd_c10 = sm.loc[(sm.cell == "C10") & (sm.init == "random") & (sm.method == "gd_best8")].set_index("n_a")
keys = ["train/field_mse_final", "val/frac_005_n5", "val/frac_005_n10", "val/med_err_x_n10"]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for name in ("operator", "optimizer"):
    cv = snaps[name].curves()
    for ax, k in zip(axes, keys):
        d = cv.loc[cv.key == k].sort_values("step")
        ax.plot(d.step, d.value, color=STYLE[name]["color"], lw=2, label=name)
        ax.set_title(k, fontsize=9)
        ax.set_xlabel("step")
        ax.grid(alpha=0.25, lw=0.5)
for ax, k, n in [(axes[1], "share", 5), (axes[2], "share", 10), (axes[3], "med_err_x", 10)]:
    ax.axhline(gd_c10.loc[n, k], color=STYLE["gd_best8"]["color"], ls="--", lw=1, label="gd best of 8 (test pairs)")
axes[0].set_yscale("log")
axes[3].set_yscale("log")
axes[1].legend(fontsize=8, frameon=False)
fig.tight_layout()

### 1d examples (C10, random start)

Per panel, A's frame: A's function (grey line), A's n_A context points (black), B placed by the truth (light blue:
it lies on A's function), the start (light grey) and the field after the method (coloured). Underneath: each B point's
position x̂ over the iterations (gd: every 30 steps) against its true position T(u) (thin blue line at the end).

In [ ]:
def show_pair(ax, ax_t, name, cell="C10", n=5, pair=0):
    pts, trs = snaps[name].table("points"), snaps[name].table("traj")
    p = pts.loc[(pts.cell == cell) & (pts.pair == pair)]
    B = p.loc[p.role == "B"].sort_values("i")
    A = p.loc[(p.role == "A") & (p.i < n)]
    Q = p.loc[p.role == "A_test"].sort_values("pos")
    tr = trs.loc[(trs.cell == cell) & (trs.n_a == n) & (trs.pair == pair)]
    t0, t1 = tr.loc[tr.t == tr.t.min()].sort_values("i"), tr.loc[tr.t == tr.t.max()].sort_values("i")
    ax.plot(Q.pos, Q.y_true, color="0.6", lw=1)
    ins = B.loc[B.flag]
    ax.scatter(ins.x_true, ins.y_true, s=10, color="#86b6ef", label="B, truth (B_inA)")
    ax.scatter(t0.x, t0.y, s=8, color="0.8", label="start")
    ax.scatter(t1.x, t1.y, s=10, color=STYLE[name]["color"], label=f"{name}, final")
    ax.scatter(A.pos, A.y, s=36, color="k", zorder=5, label="A's context")
    err = np.sqrt(np.mean((t1.x.to_numpy() - B.x_true.to_numpy())[B.flag.to_numpy()] ** 2))
    ax.set_xlim(-0.2, 1.2)
    ax.set_title(f"{name}: pair {pair}, n_A = {n}, err_x = {err:.3f}", fontsize=9)
    for _, g in tr.groupby("i"):
        ax_t.plot(g.t, g.x, color=STYLE[name]["color"], lw=0.5, alpha=0.5)
    ax_t.scatter(np.full(len(ins), tr.t.max() * 1.03), ins.x_true, s=3, color="#2a78d6")
    ax_t.set_ylabel("x̂", fontsize=8)
    ax_t.set_xlabel("iteration (gd: step)", fontsize=8)


for n in (5, 10):
    for pair in range(3):
        fig, axes = plt.subplots(2, 3, figsize=(15, 5), gridspec_kw={"height_ratios": [3, 2]})
        for j, name in enumerate(["gd", "operator", "optimizer"]):
            show_pair(axes[0, j], axes[1, j], name, n=n, pair=pair)
        axes[0, 0].legend(fontsize=7, frameon=False)
        fig.tight_layout()

## Observations

(after the runs)

## Verdict

In [ ]:
verdict(sm)